# [EJERCICIO 7]: Registro de operaciones (logging)

Notebook de pruebas para el ejercicio 7 (logging).

- **7.A–7.B:** `src/log_operaciones.py` (`current_date`, `log`).
- **7.C:** `src/insercion.py` (`insert_record`).
- **7.D:** `src/busquedas.py` (`actualizar_registros`: log `UPDATE` y cantidad de registros afectados).
- **7.E:** `src/eliminaciones.py` (p. ej. `eliminar_por_identificador`: log `DELETE` y cantidad eliminada).
- **7.F:** mismos módulos: ante error se registra `0 registros | ERROR`.

## Setup

Agregamos la raíz del proyecto al `sys.path` (igual que en otros notebooks) para poder usar imports `from src....`

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.log_operaciones import current_date, log

## Ejercicio 7.A — `current_date()`

Tiene que devolver la fecha y hora actual como string con formato `YYYY-MM-DD HH:MM:SS` (zona horaria UTC-3).

In [3]:
fecha = current_date()
print("current_date() ->", fecha)

current_date() -> 2026-05-04 09:52:24


## Ejercicio 7.B — `log()`

Escribe en `logs/operations.log` una línea por operación: fecha (con `current_date()`), nombre del dataset, tipo (`INSERT`, `UPDATE`, `DELETE`), cantidad de registros y, si hubo error, `ERROR`.

Acá probamos que se crea el archivo, que agrega líneas sin borrar las anteriores y leemos el resultado con `open()`.

In [5]:
# Desde notebooks/ subimos al proyecto
LOG_FILE = Path.cwd().parent / "logs" / "operations.log"

# Dos llamadas seguidas: la segunda línea se agrega al final sin borrar la primera
log("inaturalist", "INSERT", 3)
log("inaturalist", "UPDATE", 0, status="ERROR")

with open(LOG_FILE, encoding="utf-8") as f:
    print(f.read())

2026-05-04 09:52:37 | inaturalist | INSERT | 3 registros
2026-05-04 09:52:37 | inaturalist | UPDATE | 0 registros | ERROR



## Ejercicio 7.C — Inserción y log

`insert_record` en `src/insercion.py` debe registrar en el log cada inserción exitosa (`INSERT` y cantidad de registros).

Esa función pide los datos con `input()` (teclado). En la celda de abajo, al ejecutarla, te va a ir preguntando cada campo: podés copiar y pegar los valores de ejemplo de la misma celda, en el orden en que aparecen.

**NOTA:** el `observations.csv` de `raw_datasets` es enorme, por lo que la celda copia solo el encabezado y unas pocas filas a `processed_datasets/` para que la prueba sea rápida. El esquema es el mismo que el original.

In [6]:
from src.insercion import insert_record

RAW = ROOT / "raw_datasets" / "inaturalist" / "observations.csv"
MUESTRA = ROOT / "processed_datasets" / "inaturalist" / "obs_muestra_logging.csv"
SALIDA = ROOT / "processed_datasets" / "inaturalist" / "obs_con_insert_logging.csv"
LOG_FILE = ROOT / "logs" / "operations.log"

MUESTRA.parent.mkdir(parents=True, exist_ok=True)

if not RAW.exists():
    raise FileNotFoundError(f"Falta el dataset en {RAW}")

with open(RAW, encoding="utf-8") as fin, open(MUESTRA, "w", encoding="utf-8") as fout:
    fout.write(fin.readline())
    for _ in range(8):
        linea = fin.readline()
        if not linea:
            break
        fout.write(linea)

print(
    "Cuando pida cada campo, pegá el valor que corresponde (orden: lat, lon, fecha, país, metros, taxonomía…).\n"
    "Al final va a preguntar si querés otro registro: escribí n y Enter.\n"
)
print("--- Valores de ejemplo (uno por cada pregunta) ---")
print("-34.6037")
print("-58.3816")
print("2023-05-15")
print("AR")
print("50")
print("Turdus rufiventris")
print("12345")
print("species")
print("Animalia")
print("Chordata")
print("Aves")
print("Passeriformes")
print("Turdidae")
print("Turdus")
print("n")
print("--- Fin de la guía, ahora empiezan los input() ---\n")

insert_record("inaturalist", str(MUESTRA), str(SALIDA))

print("\nArchivo generado:", SALIDA)
print("\nÚltimas líneas del log (debería verse INSERT con 1 registro si validó bien):")
with open(LOG_FILE, encoding="utf-8") as f:
    for ln in f.readlines()[-12:]:
        print(ln, end="")

Cuando pida cada campo, pegá el valor que corresponde (orden: lat, lon, fecha, país, metros, taxonomía…).
Al final va a preguntar si querés otro registro: escribí n y Enter.

--- Valores de ejemplo (uno por cada pregunta) ---
-34.6037
-58.3816
2023-05-15
AR
50
Turdus rufiventris
12345
species
Animalia
Chordata
Aves
Passeriformes
Turdidae
Turdus
n
--- Fin de la guía, ahora empiezan los input() ---


--- Ingrese los datos del nuevo registro ---
Validando registro
Evaluando inconsistencias en las coordenadas...
Evaluando cotas de coordenadas (America del sur) del dataset...
Evaluando fechas del dataset...
Evaluando errores en el campo 'countryCode' del dataset...
Evaluando errores en el campo 'coordinateUncertainyInMeters' del dataset...
Buscando errores en los datos taxonomicos...
Registro validado exitosamente.
Creando nuevo archivo y copiando datos originales
Anexando 1 nuevos registros al final
Archivo generado en: c:\Users\rocco\OneDrive\Desktop\Facultad\2026\[PYTHON] Seminario de Le

## Ejercicio 7.D — Actualización y log

`actualizar_registros` en `src/busquedas.py` debe registrar en el log cada actualización exitosa con tipo `UPDATE` y la cantidad de registros afectados (si el mismo `id` apareciera más de una vez, serían varios; en el ejemplo es 1).

La celda de abajo arma una muestra chica desde `raw_datasets` (igual que en 7.C), toma el primer `id`, cambia solo el campo `occurrenceRemarks` y escribe el resultado en `processed_datasets/`. Al final mostramos las últimas líneas de `operations.log`.

In [8]:
import csv
from src.busquedas import actualizar_registros

RAW = ROOT / "raw_datasets" / "inaturalist" / "observations.csv"
ENT = ROOT / "processed_datasets" / "inaturalist" / "obs_muestra_update.csv"
SAL = ROOT / "processed_datasets" / "inaturalist" / "obs_salida_update.csv"
LOG_FILE = ROOT / "logs" / "operations.log"

ENT.parent.mkdir(parents=True, exist_ok=True)

if not RAW.exists():
    raise FileNotFoundError(f"Falta el dataset en {RAW}")

with open(RAW, encoding="utf-8") as fin, open(ENT, "w", encoding="utf-8") as fout:
    fout.write(fin.readline())
    for _ in range(8):
        linea = fin.readline()
        if not linea:
            break
        fout.write(linea)

with open(ENT, encoding="utf-8") as f:
    primera = next(csv.DictReader(f))

id_prueba = primera["id"]
print("id a actualizar:", id_prueba)

actualizar_registros(
    str(ENT),
    str(SAL),
    id_prueba,
    "id",
    {"occurrenceRemarks": "prueba-notebook-7D"},
    dataset="inaturalist",
)

print("\nArchivo de salida:", SAL)
print("\nÚltimas líneas del log (debería aparecer UPDATE | 1 registros):")
with open(LOG_FILE, encoding="utf-8") as f:
    for ln in f.readlines()[-12:]:
        print(ln, end="")

id a actualizar: 42369866
Registro '42369866' actualizado con exito.

Archivo de salida: c:\Users\rocco\OneDrive\Desktop\Facultad\2026\[PYTHON] Seminario de Lenguajes\TP Integrador I\processed_datasets\inaturalist\obs_salida_update.csv

Últimas líneas del log (debería aparecer UPDATE | 1 registros):
2026-05-04 09:52:37 | inaturalist | INSERT | 3 registros
2026-05-04 09:52:37 | inaturalist | UPDATE | 0 registros | ERROR
2026-05-04 09:53:07 | inaturalist | INSERT | 1 registros
2026-05-04 09:54:28 | inaturalist | UPDATE | 1 registros


## Ejercicio 7.E — Eliminación y log

En `src/eliminaciones.py`, las funciones de borrado (`eliminar_por_identificador`, `eliminar_por_lista`, `eliminar_por_condicion`, etc.) deben registrar en el log la operación con tipo `DELETE` y la cantidad de registros eliminados.

Acá probamos `eliminar_por_identificador`: copiamos una muestra desde `raw_datasets`, borramos una fila por `id` y guardamos el resultado en `processed_datasets/`. En el log debería verse `DELETE | 1 registros`.

In [10]:
import csv
from src.eliminaciones import eliminar_por_identificador

RAW = ROOT / "raw_datasets" / "inaturalist" / "observations.csv"
ENT = ROOT / "processed_datasets" / "inaturalist" / "obs_muestra_delete.csv"
SAL = ROOT / "processed_datasets" / "inaturalist" / "obs_salida_delete.csv"
LOG_FILE = ROOT / "logs" / "operations.log"

ENT.parent.mkdir(parents=True, exist_ok=True)

if not RAW.exists():
    raise FileNotFoundError(f"Falta el dataset en {RAW}")

with open(RAW, encoding="utf-8") as fin, open(ENT, "w", encoding="utf-8") as fout:
    fout.write(fin.readline())
    for _ in range(8):
        linea = fin.readline()
        if not linea:
            break
        fout.write(linea)

with open(ENT, encoding="utf-8") as f:
    primera = next(csv.DictReader(f))

id_borrar = primera["id"]
print("id a eliminar:", id_borrar)

eliminar_por_identificador(
    str(ENT),
    str(SAL),
    "id",
    id_borrar,
    dataset="inaturalist",
)

print("\nArchivo sin ese registro:", SAL)
print("\nÚltimas líneas del log (debería aparecer DELETE | 1 registros):")
with open(LOG_FILE, encoding="utf-8") as f:
    for ln in f.readlines()[-12:]:
        print(ln, end="")

id a eliminar: 42369866
Registro encontrado correctamente.

Archivo sin ese registro: c:\Users\rocco\OneDrive\Desktop\Facultad\2026\[PYTHON] Seminario de Lenguajes\TP Integrador I\processed_datasets\inaturalist\obs_salida_delete.csv

Últimas líneas del log (debería aparecer DELETE | 1 registros):
2026-05-04 09:52:37 | inaturalist | INSERT | 3 registros
2026-05-04 09:52:37 | inaturalist | UPDATE | 0 registros | ERROR
2026-05-04 09:53:07 | inaturalist | INSERT | 1 registros
2026-05-04 09:54:28 | inaturalist | UPDATE | 1 registros
2026-05-04 09:56:15 | inaturalist | DELETE | 1 registros


## Ejercicio 7.F — Errores también en el log

Si algo falla, el log debe mostrar 0 registros y el sufijo `ERROR` (como en el ejemplo de la consigna).

Abajo forzamos tres casos: `insert_record` con un nombre de dataset que no existe; `actualizar_registros` y `eliminar_por_identificador` con un `id` que no está en la muestra.

In [11]:
from src.insercion import insert_record
from src.busquedas import actualizar_registros
from src.eliminaciones import eliminar_por_identificador

RAW = ROOT / "raw_datasets" / "inaturalist" / "observations.csv"
MUESTRA = ROOT / "processed_datasets" / "inaturalist" / "obs_muestra_errores.csv"
SAL_UPDATE = ROOT / "processed_datasets" / "inaturalist" / "obs_salida_err_update.csv"
SAL_DELETE = ROOT / "processed_datasets" / "inaturalist" / "obs_salida_err_delete.csv"
LOG_FILE = ROOT / "logs" / "operations.log"

MUESTRA.parent.mkdir(parents=True, exist_ok=True)

if not RAW.exists():
    raise FileNotFoundError(f"Falta el dataset en {RAW}")

with open(RAW, encoding="utf-8") as fin, open(MUESTRA, "w", encoding="utf-8") as fout:
    fout.write(fin.readline())
    for _ in range(8):
        linea = fin.readline()
        if not linea:
            break
        fout.write(linea)

print("1) INSERT con dataset inexistente:")
insert_record("dataset_que_no_existe", str(MUESTRA), str(MUESTRA))

print("\n2) UPDATE con id inexistente:")
actualizar_registros(
    str(MUESTRA),
    str(SAL_UPDATE),
    "99999999999999999",
    "id",
    {"occurrenceRemarks": "no-importa"},
    dataset="inaturalist",
)

print("\n3) DELETE con id inexistente:")
eliminar_por_identificador(
    str(MUESTRA),
    str(SAL_DELETE),
    "id",
    "99999999999999999",
    dataset="inaturalist",
)

print("\nÚltimas líneas del log (tres líneas con | ERROR):")
with open(LOG_FILE, encoding="utf-8") as f:
    for ln in f.readlines()[-15:]:
        print(ln, end="")

1) INSERT con dataset inexistente:
Dataset 'dataset_que_no_existe' no reconocido.

2) UPDATE con id inexistente:

3) DELETE con id inexistente:
Registro no encontrado.

Últimas líneas del log (tres líneas con | ERROR):
2026-05-04 09:52:37 | inaturalist | INSERT | 3 registros
2026-05-04 09:52:37 | inaturalist | UPDATE | 0 registros | ERROR
2026-05-04 09:53:07 | inaturalist | INSERT | 1 registros
2026-05-04 09:54:28 | inaturalist | UPDATE | 1 registros
2026-05-04 09:56:15 | inaturalist | DELETE | 1 registros
2026-05-04 09:56:21 | dataset_que_no_existe | INSERT | 0 registros | ERROR
2026-05-04 09:56:21 | inaturalist | UPDATE | 0 registros | ERROR
2026-05-04 09:56:21 | inaturalist | DELETE | 0 registros | ERROR
